# Metric Review

In [2]:
import os
import re
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
if root.name == "ipynb":
    root = root.parent
    os.chdir(root)


In [8]:
for c in pd.read_csv("metrics/biapy/biapy-channel-scale-seunet/results/biapy-channel-scale-seunet_0/test_results_metrics.csv").columns: print(c)

file
iou (f channel)
l1 (dc channel)
l1 (dn channel)
iou (p channel)
0.3 TH fp
0.3 TH tp
0.3 TH fn
0.3 TH precision
0.3 TH recall
0.3 TH accuracy
0.3 TH f1
0.3 TH n_true
0.3 TH n_pred
0.3 TH mean_true_score
0.3 TH mean_matched_score
0.3 TH panoptic_quality
0.5 TH fp
0.5 TH tp
0.5 TH fn
0.5 TH precision
0.5 TH recall
0.5 TH accuracy
0.5 TH f1
0.5 TH n_true
0.5 TH n_pred
0.5 TH mean_true_score
0.5 TH mean_matched_score
0.5 TH panoptic_quality
0.75 TH fp
0.75 TH tp
0.75 TH fn
0.75 TH precision
0.75 TH recall
0.75 TH accuracy
0.75 TH f1
0.75 TH n_true
0.75 TH n_pred
0.75 TH mean_true_score
0.75 TH mean_matched_score
0.75 TH panoptic_quality
0.3 TH (post)fp
0.3 TH (post)tp
0.3 TH (post)fn
0.3 TH (post)precision
0.3 TH (post)recall
0.3 TH (post)accuracy
0.3 TH (post)f1
0.3 TH (post)n_true
0.3 TH (post)n_pred
0.3 TH (post)mean_true_score
0.3 TH (post)mean_matched_score
0.3 TH (post)panoptic_quality
0.5 TH (post)fp
0.5 TH (post)tp
0.5 TH (post)fn
0.5 TH (post)precision
0.5 TH (post)recall
0.5 

In [ ]:
_NUMPY_REPR = re.compile(r"^np\.\w+\((.*)\)$")


def _clean_value(value):
    """Strip numpy repr strings (e.g. ``np.float64(0.1)``) to plain numbers."""
    if isinstance(value, str):
        match = _NUMPY_REPR.match(value.strip())
        if match:
            inner = match.group(1)
            try:
                return int(inner) if inner.lstrip("-").isdigit() else float(inner)
            except ValueError:
                return inner
        return value
    if isinstance(value, list):
        return [_clean_value(v) for v in value]
    if isinstance(value, dict):
        return {k: _clean_value(v) for k, v in value.items()}
    return value


def _sample_stem(path: Path) -> str:
    """Derive sample stem from an eval-inst TOML filename."""
    name = path.name
    if name.endswith(".zarr.toml"):
        return name[: -len(".zarr.toml")]
    marker = "_volumes_"
    if marker in name:
        return name.split(marker, 1)[0]
    return path.stem


def _pick_toml_files(eval_inst_dir: Path) -> list[Path]:
    """One TOML per sample; prefer ``*_volumes_*`` over ``*.zarr.toml`` duplicates."""
    by_stem: dict[str, Path] = {}
    for path in sorted(eval_inst_dir.glob("*.toml")):
        stem = _sample_stem(path)
        prev = by_stem.get(stem)
        if prev is None:
            by_stem[stem] = path
            continue
        # Prefer the evaluate_file output over biapy_eval's ``*.zarr.toml`` dump.
        if prev.name.endswith(".zarr.toml") and "_volumes_" in path.name:
            by_stem[stem] = path
    return list(by_stem.values())


def _flatten_toml(data: dict, stem: str, path: Path) -> dict:
    """Flatten ``[general]`` + ``[confusion_matrix]`` into one row of scalars."""
    row: dict = {"stem": stem, "file": path.name}
    for section in ("general", "confusion_matrix"):
        block = data.get(section, {})
        for key, value in block.items():
            value = _clean_value(value)
            if isinstance(value, dict):
                for sub_key, sub_val in value.items():
                    row[f"{key}.{sub_key}"] = _clean_value(sub_val)
            elif isinstance(value, list):
                continue  # skip per-instance lists (e.g. gt_skel_coverage)
            else:
                row[key] = value
    return row


def load_eval_inst_metrics(eval_inst_dir) -> pd.DataFrame:
    """Compile all eval-inst TOML metrics in a folder into a DataFrame.

    Parameters
    ----------
    eval_inst_dir :
        Path to an ``eval-inst_metrics/{split}`` directory, e.g.
        ``metrics/biapy/biapy-v1-no-aug/results/biapy-v1-no-aug_0/eval-inst_metrics/test``.

    Returns
    -------
    pd.DataFrame
        One row per sample with flattened general + confusion-matrix metrics.
    """
    eval_inst_dir = Path(eval_inst_dir)
    if not eval_inst_dir.is_dir():
        raise FileNotFoundError(f"eval-inst dir not found: {eval_inst_dir}")

    rows = []
    for path in _pick_toml_files(eval_inst_dir):
        with path.open("rb") as fh:
            data = tomllib.load(fh)
        rows.append(_flatten_toml(data, _sample_stem(path), path))

    if not rows:
        raise FileNotFoundError(f"No .toml files under {eval_inst_dir}")

    return pd.DataFrame(rows).sort_values("stem").reset_index(drop=True)


df = load_eval_inst_metrics(
    "metrics/biapy/biapy-v1-no-aug/results/biapy-v1-no-aug_0/eval-inst_metrics/test"
)
df

,stem,file,Num GT,Num Pred,TP_05,TP_05_rel,avg_TP_05_cldice,avg_gt_skel_coverage,avg_f1_cov_score,FM,...,th_0_85.false_merge,th_0_95.AP_TP,th_0_95.AP_FP,th_0_95.AP_FN,th_0_95.precision,th_0_95.recall,th_0_95.AP,th_0_95.fscore,th_0_95.false_split,th_0_95.false_merge
0,JRC_SS04989-20160318_24_A2,JRC_SS04989-20160318_24_A2_volumes_pred_instan...,3,1,0,0.0,0.0,0.000000,0.055556,2,...,2,0,1,3,0.0,0.0,0.0,0.0,0,2
1,R14A02-20180905_65_A6,R14A02-20180905_65_A6_volumes_pred_instance_cl...,7,4,0,0.0,0.0,0.062733,0.081872,6,...,3,0,4,7,0.0,0.0,0.0,0.0,2,2
2,R54A09-20181019_64_H1,R54A09-20181019_64_H1_volumes_pred_instance_cl...,1,4,0,0.0,0.0,0.000000,0.066667,0,...,0,0,4,1,0.0,0.0,0.0,0.0,0,0
3,VT011145-20171222_63_I1,VT011145-20171222_63_I1_volumes_pred_instance_...,9,1,0,0.0,0.0,0.000000,0.022222,8,...,6,0,1,9,0.0,0.0,0.0,0.0,0,5
4,VT027175-20171031_62_H3,VT027175-20171031_62_H3_volumes_pred_instance_...,3,1,0,0.0,0.0,0.000000,0.027778,2,...,0,0,1,3,0.0,0.0,0.0,0.0,0,0
5,VT027175-20171031_62_H4,VT027175-20171031_62_H4_volumes_pred_instance_...,6,4,0,0.0,0.0,0.000000,0.022222,5,...,2,0,4,6,0.0,0.0,0.0,0.0,0,1
6,VT050157-20171110_61_C1,VT050157-20171110_61_C1_volumes_pred_instance_...,5,1,0,0.0,0.0,0.000000,0.037037,3,...,1,0,1,5,0.0,0.0,0.0,0.0,0,1


In [14]:
CONFIG_STEM = "biapy-v1-no-aug"
RUN="0"
df = load_eval_inst_metrics(
    f"metrics/biapy/{CONFIG_STEM}/results/{CONFIG_STEM}_0/eval-inst_metrics/test"
)
df.loc[:, ['stem', 'Num GT', 'Num Pred', 'FM', 'FS']]


,stem,Num GT,Num Pred,FM,FS
0,JRC_SS04989-20160318_24_A2,3,1,2,0
1,R14A02-20180905_65_A6,7,4,6,2
2,R54A09-20181019_64_H1,1,4,0,0
3,VT011145-20171222_63_I1,9,1,8,0
4,VT027175-20171031_62_H3,3,1,2,0
5,VT027175-20171031_62_H4,6,4,5,0
6,VT050157-20171110_61_C1,5,1,3,0


In [17]:
CONFIG_STEM = "biapy-v1-channel-rot"
RUN="0"
df = load_eval_inst_metrics(
    f"metrics/biapy/{CONFIG_STEM}/results/{CONFIG_STEM}_0/eval-inst_metrics/test"
)
df.loc[:, ['stem', 'Num GT', 'Num Pred', 'FM', 'FS']]


FileNotFoundError: No .toml files under metrics/biapy/biapy-v1-channel-rot/results/biapy-v1-channel-rot_0/eval-inst_metrics/test

In [16]:
CONFIG_STEM = "biapy-v2-no-aug"
RUN="0"
df = load_eval_inst_metrics(
    f"metrics/biapy/{CONFIG_STEM}/results/{CONFIG_STEM}_0/eval-inst_metrics/test"
)
df.loc[:, ['stem', 'Num GT', 'Num Pred', 'FM', 'FS']]


,stem,Num GT,Num Pred,FM,FS
0,JRC_SS04989-20160318_24_A2,3,123,0,0
1,R14A02-20180905_65_A6,7,34,2,0
2,R54A09-20181019_64_H1,1,58,0,0
3,VT011145-20171222_63_I1,9,27,4,5
4,VT027175-20171031_62_H3,3,140,0,2
5,VT027175-20171031_62_H4,6,178,3,3
6,VT050157-20171110_61_C1,5,100,0,2
